In [1]:
from sklearn.feature_selection import SelectKBest, chi2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from google.colab import files
import io
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
uploaded = files.upload()

# Assuming only one file is uploaded and it's a CSV
df = None
if uploaded:
    filename = list(uploaded.keys())[0]
    uploaded_file_content = uploaded[filename]
    print(f'User uploaded file "{filename}" with length {len(uploaded_file_content)} bytes')

    # Read the byte content into a pandas DataFrame, assuming CSV format
    try:
        df = pd.read_csv(io.BytesIO(uploaded_file_content))
        print(f'Successfully loaded "{filename}" into a pandas DataFrame.')
        display(df.head()) # Display the first 5 rows to confirm
    except Exception as e:
        print(f'Error reading file into DataFrame: {e}')
else:
    print('No file was uploaded.')

Saving processed_uploaded_data.csv to processed_uploaded_data.csv
User uploaded file "processed_uploaded_data.csv" with length 13319857 bytes
Successfully loaded "processed_uploaded_data.csv" into a pandas DataFrame.


,HighBP,HighChol,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,GenHlth,MentHlth,PhysHlth,DiffWalk,Age,Education,Income,Diabetes_binary
0,0.0,0.0,28.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0,4.0,5.0,0.0
1,1.0,0.0,23.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,13.0,4.0,7.0,0.0
2,1.0,1.0,29.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,9.0,6.0,8.0,0.0
3,1.0,1.0,39.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,7.0,4.0,7.0,0.0
4,0.0,1.0,16.0,1.0,0.0,0.0,1.0,0.0,5.0,30.0,30.0,1.0,7.0,5.0,1.0,0.0


In [4]:
# Identify numerical & categorical features

numerical_features = [
    'BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Income'
]
categorical_features_for_ohe = [
    'HighBP', 'HighChol', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
    'PhysActivity', 'HvyAlcoholConsump', 'DiffWalk', 'Education'
]

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_for_ohe)
    ])

pdx_processed = preprocessor.fit_transform(df)


# get_feature_names_out method:  returns the column names in the order they appear from the numerical and one-hot encoded features.

# The order of the feature names must match the order from preprocessor
# get numerical feature names first, next categorical feature names
feature_names = numerical_features + \
                list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features_for_ohe))

# Convert processed array --> DataFrame
pdx_final_df = pd.DataFrame(pdx_processed, columns=feature_names, index=df.index)

print("Original features (first 5 rows of numerical and categorical):")
display(df[numerical_features + categorical_features_for_ohe].head())

print("\nProcessed features (first 5 rows of scaled numerical and one-hot encoded categorical):")
display(pdx_final_df.head())

print(f"\nNumber of features before processing: {df.shape[1]}")
print(f"Number of features after processing and one-hot encoding: {pdx_final_df.shape[1]}")

Original features (first 5 rows of numerical and categorical):


,BMI,GenHlth,MentHlth,PhysHlth,Age,Income,HighBP,HighChol,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,HvyAlcoholConsump,DiffWalk,Education
0,28.0,2.0,0.0,0.0,2.0,5.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.0
1,23.0,2.0,0.0,0.0,13.0,7.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.0
2,29.0,1.0,0.0,0.0,9.0,8.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,6.0
3,39.0,4.0,0.0,0.0,7.0,7.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
4,16.0,5.0,30.0,30.0,7.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,5.0



Processed features (first 5 rows of scaled numerical and one-hot encoded categorical):


,BMI,GenHlth,MentHlth,PhysHlth,Age,Income,HighBP_0.0,HighBP_1.0,HighChol_0.0,HighChol_1.0,...,HvyAlcoholConsump_0.0,HvyAlcoholConsump_1.0,DiffWalk_0.0,DiffWalk_1.0,Education_1.0,Education_2.0,Education_3.0,Education_4.0,Education_5.0,Education_6.0
0,-0.057282,-0.479079,-0.430031,-0.487165,-1.977081,-0.506998,1.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-0.815055,-0.479079,-0.430031,-0.487165,1.627845,0.457948,0.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.094273,-1.415079,-0.430031,-0.487165,0.316963,0.940421,0.0,1.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.609820,1.392923,-0.430031,-0.487165,-0.338479,0.457948,0.0,1.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-1.875938,2.328923,3.614406,2.950984,-0.338479,-2.436889,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0



Number of features before processing: 16
Number of features after processing and one-hot encoding: 28


In [ ]:
output_filename = 'scaled_data.csv'
pdx_final_df.to_csv(output_filename, index=False)
files.download(output_filename)